# 00 - MLflow connection test

Run this once, before your first real experiment. It checks that your Colab can talk to the team's MLflow server on DagsHub.

**Before running:** add three secrets in Colab (key icon, left sidebar) and turn on *Notebook access* for each:

| Secret | Value |
|---|---|
| `MLFLOW_TRACKING_URI` | `https://dagshub.com/MOHAED-ABDALFATTAH/graduation-ml-framework.mlflow` |
| `MLFLOW_USERNAME` | your DagsHub username |
| `MLFLOW_PASSWORD` | your DagsHub access token (never your password) |

Set `OWNER` in the first code cell to your own name.

In [ ]:
OWNER = "your_name"   # <-- change this

!pip install -q mlflow

import os
from google.colab import userdata

os.environ["MLFLOW_TRACKING_URI"] = userdata.get("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_PASSWORD")
print("Tracking URI:", os.environ["MLFLOW_TRACKING_URI"])
print("Username:", os.environ["MLFLOW_TRACKING_USERNAME"])

## 1. Parameters and metrics

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("Connection_Test")

with mlflow.start_run(run_name=f"connection_test_params_{OWNER}") as run:
    mlflow.set_tags({"owner": OWNER, "purpose": "connection_test"})
    mlflow.log_params({"learning_rate": 0.001, "batch_size": 32})
    for step in range(5):
        mlflow.log_metric("fake_loss", 1.0 / (step + 1), step=step)
    print("Run ID:", run.info.run_id)
print("Params and metrics logged OK")

## 2. Artifacts (the stricter test)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [1, 4, 9, 16])
ax.set_title("Test plot")
fig.savefig("test_plot.png")

with open("test_notes.txt", "w") as f:
    f.write("artifact upload test")

with mlflow.start_run(run_name=f"connection_test_artifacts_{OWNER}"):
    mlflow.set_tag("owner", OWNER)
    mlflow.log_artifact("test_plot.png")
    mlflow.log_artifact("test_notes.txt")
print("Artifacts logged OK")

## 3. Framework helper test

Clones the team repo and uses the shared tracking helper (standard tags + failure logging).

In [ ]:
import os, sys

if not os.path.exists("graduation-ml-framework"):
    !git clone -q https://github.com/MOHAED-ABDALFATTAH/graduation-ml-framework.git
%cd graduation-ml-framework
!git pull -q
!pip install -q -e .

from src.tracking import setup_tracking, start_tracked_run, log_metrics_safe

setup_tracking("Connection_Test")
with start_tracked_run(f"helper_test_{OWNER}", owner=OWNER, track="test",
                       dataset_version="none", seed=42,
                       config={"training": {"lr": 0.001}}):
    log_metrics_safe({"dummy_map": 0.5})
print("Framework helper OK")

## Check the result

Open the DagsHub repo, click the **Experiments** tab, and look for the `Connection_Test` experiment. You should see three runs with your name. Then delete them (select, then *Delete*) so they don't count toward the free plan's limit.